# Demo 04 — Relational Linkage with CTGAN

This notebook demonstrates **multi-table synthetic data generation** with foreign-key linkage. The workflow:

1. Generate parent (users) and child (orders) tables with a foreign key
2. Train a CTGAN on the parent table
3. Train a `LinkageModel` to learn the child-count distribution per parent
4. Train a conditional CTGAN on the child table using parent context columns
5. Generate synthetic parents, sample child counts, and generate linked children

No external data files or Spark are required.

In [ ]:
import numpy as np
import pandas as pd

from syntho_hive.interface.config import Metadata
from syntho_hive.core.models.ctgan import CTGAN
from syntho_hive.relational.linkage import LinkageModel

## 1. Generate Parent-Child Seed Data

We create 300 parent rows (users) and a variable number of child rows (orders) per parent (0 to 5).

In [ ]:
def make_parent_child_seed_data(num_parents: int = 300, max_children: int = 5):
    """Create a small parent/child dataset to train linkage + CTGAN."""
    rng = np.random.default_rng(10)
    regions = ["NE", "SE", "MW", "W"]

    parents = pd.DataFrame(
        {
            "user_id": np.arange(1, num_parents + 1),
            "region": rng.choice(regions, size=num_parents, p=[0.3, 0.2, 0.25, 0.25]),
            "age": rng.integers(20, 70, size=num_parents),
        }
    )

    child_rows = []
    order_id = 1
    for _, row in parents.iterrows():
        n_orders = rng.integers(0, max_children + 1)
        for _ in range(n_orders):
            child_rows.append(
                {
                    "order_id": order_id,
                    "user_id": row["user_id"],
                    "basket_value": max(5, rng.normal(80, 25)),
                    "channel": rng.choice(
                        ["web", "store", "mobile"], p=[0.5, 0.3, 0.2]
                    ),
                }
            )
            order_id += 1

    children = pd.DataFrame(child_rows)
    return parents, children

parent_df, child_df = make_parent_child_seed_data()
print(f"Parents: {parent_df.shape}, Children: {child_df.shape}")
print(f"Average orders per user: {len(child_df) / len(parent_df):.1f}")
parent_df.head()

## 2. Define Relational Metadata

The metadata declares the primary keys, the foreign key relationship (`orders.user_id -> users.user_id`), and which parent columns to use as context when generating children.

In [ ]:
meta = Metadata()
meta.add_table(
    name="users", pk="user_id", pii_cols=[], high_cardinality_cols=["region"]
)
meta.add_table(
    name="orders",
    pk="order_id",
    fk={"user_id": "users.user_id"},
    parent_context_cols=["region"],
    constraints={"basket_value": {"dtype": "float", "min": 1.0}},
)

## 3. Train the Parent CTGAN

We train a CTGAN on the users table. This model learns the marginal and joint distributions of `region` and `age`.

In [ ]:
users_model = CTGAN(
    meta,
    batch_size=128,
    epochs=3,
    generator_dim=(128, 128),
    discriminator_dim=(128, 128),
    embedding_dim=64,
)
print("Training users CTGAN...")
users_model.fit(parent_df, table_name="users")
print("Users model training complete.")

## 4. Train the Linkage Model

The `LinkageModel` learns how many children each parent has, so we can reproduce the cardinality distribution during generation.

In [ ]:
linkage = LinkageModel()
print("Training linkage model...")
linkage.fit(parent_df, child_df, fk_col="user_id", pk_col="user_id")
print("Linkage model training complete.")

## 5. Train the Child CTGAN with Parent Context

We join the parent `region` column onto the child table and pass it as context during training. This lets the child CTGAN condition on parent attributes.

In [ ]:
joined = child_df.merge(parent_df[["user_id", "region"]], on="user_id", how="left")
context_df = joined[["region"]].copy()

orders_model = CTGAN(
    meta,
    batch_size=128,
    epochs=3,
    generator_dim=(128, 128),
    discriminator_dim=(128, 128),
    embedding_dim=64,
)
print("Training orders CTGAN with parent context...")
orders_model.fit(child_df, context=context_df, table_name="orders")
print("Orders model training complete.")

## 6. Generate Synthetic Data

We generate 150 synthetic parents, sample child counts using the linkage model, then generate the corresponding children conditioned on each parent's region.

In [ ]:
num_parents = 150

# Generate synthetic parents
users = users_model.sample(num_parents)
users.insert(0, "user_id", range(1, len(users) + 1))
print(f"Generated {len(users)} synthetic users")

# Sample child counts per parent
counts = linkage.sample_counts(users)
total_children = int(counts.sum())
print(f"Total children to generate: {total_children}")

# Build context rows and FK values for children
context_rows = []
fk_values = []
for idx, parent in users.iterrows():
    repeat = counts[idx]
    if repeat <= 0:
        continue
    fk_values.extend([parent["user_id"]] * repeat)
    context_rows.extend([{"region": parent["region"]}] * repeat)

# Generate children
if total_children > 0:
    context_for_gen = pd.DataFrame(context_rows)
    orders = orders_model.sample(total_children, context=context_for_gen)
    orders.insert(0, "order_id", range(1, len(orders) + 1))
    orders["user_id"] = fk_values
else:
    orders = pd.DataFrame(columns=["order_id", "user_id", "basket_value", "channel"])

print(f"Generated {len(orders)} synthetic orders")

## 7. Inspect Results

In [ ]:
print("=== Synthetic Users ===")
print(users.head(10))

print("\n=== Synthetic Orders ===")
print(orders.head(10))

print(f"\nAverage orders per user: {len(orders) / len(users):.1f}")